In [1]:
!pip install pandas numpy scikit-learn matplotlib mlxtend

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("OnlineRetail_(1)_(1).csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-1-2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12-1-2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-1-2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-1-2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-1-2010 8:26,3.39,17850.0,United Kingdom


In [4]:
# Remove missing Customer IDs
df = df[df['CustomerID'].notnull()]

# Remove returns (Invoice numbers starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Only keep positive quantities and prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Add total price column
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']


In [5]:
# Top 10 products by sales quantity
df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)


Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54415
JUMBO BAG RED RETROSPOT               46181
WHITE HANGING HEART T-LIGHT HOLDER    36725
ASSORTED COLOUR BIRD ORNAMENT         35362
PACK OF 72 RETROSPOT CAKE CASES       33693
POPCORN HOLDER                        30931
RABBIT NIGHT LIGHT                    27202
MINI PAINT SET VINTAGE                26076
Name: Quantity, dtype: int64

In [6]:
popular_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print("Top Recommended Products:")
print(popular_products)

Top Recommended Products:
Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54415
JUMBO BAG RED RETROSPOT               46181
WHITE HANGING HEART T-LIGHT HOLDER    36725
ASSORTED COLOUR BIRD ORNAMENT         35362
PACK OF 72 RETROSPOT CAKE CASES       33693
POPCORN HOLDER                        30931
RABBIT NIGHT LIGHT                    27202
MINI PAINT SET VINTAGE                26076
Name: Quantity, dtype: int64


In [7]:
from mlxtend.frequent_patterns import apriori, association_rules

# Create basket for UK customers
basket = (df[df['Country'] =="United Kingdom"]
          .groupby(['InvoiceNo','Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

# Convert values to True/False instead of 0/1
basket = basket.gt(0)   # now values are True (bought) / False (not bought)

# Run Apriori
frequent_items = apriori(basket, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_items, metric="lift", min_threshold=1)

rules.sort_values("confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
72,"(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.023009,0.036766,0.020485,0.890339,24.216650,1.0,0.019639,8.783780,0.981284,0.521407,0.886154,0.723764
71,"(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",(ROSES REGENCY TEACUP AND SAUCER),0.024270,0.040731,0.020485,0.844059,20.723028,1.0,0.019497,6.151506,0.975418,0.460189,0.837438,0.673505
5,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.029617,0.036766,0.024270,0.819473,22.289120,1.0,0.023181,5.335669,0.984286,0.576320,0.812582,0.739802
7,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.036766,0.040731,0.028595,0.777778,19.095706,1.0,0.027098,4.316713,0.983802,0.584767,0.768342,0.739921
63,(PINK REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.029617,0.040731,0.023009,0.776876,19.073573,1.0,0.021802,4.299271,0.976492,0.486041,0.767402,0.670887
2,(GARDENERS KNEELING PAD CUP OF TEA),(GARDENERS KNEELING PAD KEEP CALM),0.037667,0.044575,0.027514,0.730463,16.387169,1.0,0.025835,3.544682,0.975729,0.502744,0.717887,0.673857
70,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.028595,0.029617,0.020485,0.716387,24.188581,1.0,0.019638,3.421500,0.986878,0.542994,0.707730,0.704035
6,(ROSES REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.040731,0.036766,0.028595,0.702065,19.095706,1.0,0.027098,3.233034,0.987869,0.584767,0.690693,0.739921
75,(PINK REGENCY TEACUP AND SAUCER),"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",0.029617,0.028595,0.020485,0.691684,24.188581,1.0,0.019638,3.150674,0.987917,0.542994,0.682608,0.704035
64,(RED HANGING HEART T-LIGHT HOLDER),(WHITE HANGING HEART T-LIGHT HOLDER),0.038568,0.113180,0.025712,0.666667,5.890304,1.0,0.021347,2.660459,0.863534,0.204004,0.624125,0.446921


In [8]:
import datetime as dt

# Step 1: Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Step 2: Now you can safely calculate reference date
NOW = df['InvoiceDate'].max() + dt.timedelta(days=1)

# Step 3: Build RFM table
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (NOW - x.max()).days,  # Recency
    'InvoiceNo': 'count',                           # Frequency
    'TotalPrice': 'sum'                             # Monetary
})

rfm.rename(columns={
    'InvoiceDate':'Recency',
    'InvoiceNo':'Frequency',
    'TotalPrice':'Monetary'
}, inplace=True)

rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,182,4310.00
12348.0,75,31,1797.24
12349.0,19,73,1757.55
12350.0,310,17,334.40


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Build a User-Item Matrix
user_item_matrix = df.pivot_table(
    index='CustomerID',
    columns='StockCode',
    values='Quantity',
    aggfunc='sum',
    fill_value=0
)

# Step 2: Compute Item-Item Similarity (cosine similarity)
item_similarity = cosine_similarity(user_item_matrix.T)

# Step 3: Put into a DataFrame for easy lookup
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

# Step 4: Function to recommend similar items
def recommend_similar_items(item_id, top_n=5):
    if item_id not in item_similarity_df.columns:
        return f"Item {item_id} not found in dataset"
    similar_items = item_similarity_df[item_id].sort_values(ascending=False)[1:top_n+1]
    return similar_items

# Example: Recommend items similar to StockCode '85123A'
recommend_similar_items('85123A')

StockCode
21175    0.749651
21733    0.658732
82552    0.643868
82551    0.642480
23288    0.630982
Name: 85123A, dtype: float64